In [18]:
from baseclasses.network import Network
from scipy.optimize import root, minimize

import pickle
import os
import numpy as np

In [46]:
# Load class instance at crash point
net_id = 'Profile 3'
Kvleak_bool = True
dis_steps = 125
c = 50
N = 27091
pump_type = 'curve'

if N is not None:
    file_name = f"{net_id}_N={N}_Kvleak={Kvleak_bool}_hsteps={dis_steps}_pump={c}kPa_{pump_type}.pkl"
else: 
    file_name = f"{net_id}_Kvleak={Kvleak_bool}_hsteps={dis_steps}_pump={c}kPa_{pump_type}.pkl"

    
# base_dir = os.path.dirname(__file__)
base_dir = os.path.abspath('')

file = os.path.join(base_dir, 'debug', file_name)

with open(file, 'rb') as f:
    state = pickle.load(f)

net = Network.__new__(Network)  # create instance without calling __init__
net.__dict__.update(state)

# Determine N of the crashed state
zero_cols = (net.mflow_all == 0).all(axis=0)
non_zero_cols = ~zero_cols  # flip: True where column is NOT all-zero
idx = non_zero_cols[::-1].argmax()  # first non-zero from the right
N = len(non_zero_cols) - 1 - idx

pipe_ids = np.array(list(net.pipes.keys()))
p = len(pipe_ids)

# Initial guess for the mflows in the pipes
# mflow0 = net.mflow_all[:,N-1] 
# mflow0[mflow0 == 0] = 0.05 # Avoid zero initial guess for better convergence, can be tuned.
mflow0 = np.ones(p)*0.1

active_graph, active_mask = net.update_valves(N)
net.build_loop_matrix(active_graph)

# Reduce and create active incidence matrix
net.incidence_matrix_red = net.incidence_matrix[1:,:]
incidence_matrix_active = net.incidence_matrix_red[:, active_mask]
net.incidence_matrix_active = incidence_matrix_active[~np.all(incidence_matrix_active == 0, axis=1), :]  # Remove zero rows (disconnected nodes)

# Adjust all vectors to active pipes
friction_vector = net.pressure_friction_vector + \
                net.Kp_array + \
                net.inv_Kv_array ** 2

                # inv_Kv_array_debug ** 2
                # np.round(net.inv_Kv_array ** 2)
net.friction_vector_active = friction_vector[active_mask]

net.pressure_elevation_vector_active = net.pressure_elevation_vector[active_mask]
net.pump_coeff_active = net.pump_coeff[active_mask, :]
mflow0_active = mflow0[active_mask]

result = root(net.res, mflow0_active, jac = net.jac, method = 'hybr', tol = 1e-5)
mflow = result.x

# Try it for different rounding of the vector

# Retry with progressively rounded friction vectors if initial solve failed

if result.success == False or (result.x > 0).all():
    # solved = False
    for precision in range(5,0,-1):

        friction_vector = net.pressure_friction_vector + \
                        net.Kp_array + \
                        np.round(net.inv_Kv_array**2,precision)
        net.friction_vector_active = friction_vector[active_mask]
        
        # Solves system of non linear equations using scipy root function
        result = root(net.res, mflow0_active, jac = net.jac, method = 'hybr', tol = 1e-6)

        # Try other root solver
        lm = False
        if result.success == False or (result.x < 0).all():
            result = root(net.res, mflow0_active, jac = net.jac, method = 'lm', tol = 1e-6)
            lm = True

        if result.success == True and (result.x > 0).all():
            solved = True
            print(f'precision {i}')
            if lm:
                print('used Levenbergh Marquardt')
            break
    
    # In case root solving didn't work.
    if not solved: 
        for precision in range(5,0,-1): 
            friction_vector = net.pressure_friction_vector + \
                            net.Kp_array + \
                            np.round(net.inv_Kv_array**2,precision)
            net.friction_vector_active = friction_vector[active_mask]

            result = minimize(
                fun=lambda x: np.sum(net.res(x)**2),
                x0=mflow0_active,
                jac=lambda x: 2 * net.jac(x).T @ net.res(x),  # chain rule: 2 * J^T * F
                method='L-BFGS-B',
                tol=1e-10
            )

            if result.success == True and (result.x > 0).all():
                print(f'timestep {N} used minimization')
                break


elif (result.x > 0).all():
    print(f'Result contains negative flows')

print(result.message)
print(result.success)
print(f'residual sum of result.x {sum(net.res(mflow))}')

# print('mflow0:', mflow0_active)
# print('N:', N)
# print('friction:', sum(net.friction_vector_active))
# print(f'pressure friction {sum(net.pressure_friction_vector)}')
# print(f'Kp array {sum(net.Kp_array)}')
# print('inv_Kv:', sum(net.inv_Kv_array))
# print('loop_matrix shape:', net.loop_matrix_active.shape)
# print('incidence shape:', net.incidence_matrix_active.shape)
# print(f' residual sum saved in net.mflow_all {np.sum(net.res(net.mflow_all[active_mask,N]))}')
# print(f' sum in net.mflow_all {sum(net.mflow_all[:,N])}')
# print(f'Are they equal {result.x == net.mflow_all[active_mask,N]}')
# print(result.x)
# print(f' mflow_all N active_mask {net.mflow_all[active_mask,N]}')

result

precision 5
The solution converged.
True
residual sum of result.x 2.0254053578560125


 message: The solution converged.
 success: True
  status: 1
     fun: [ 0.000e+00  0.000e+00 ... -5.708e-04  3.515e-04]
       x: [ 9.493e-01  9.493e-01 ...  9.630e-03  9.630e-03]
    nfev: 168
    njev: 10
    fjac: [[-1.282e-13  1.719e-14 ... -2.169e-01 -2.667e-01]
           [ 3.173e-03  3.158e-11 ...  8.628e-01  2.794e-01]
           ...
           [-3.012e-03  1.426e-02 ... -1.758e-06  1.505e-05]
           [-5.723e-04  2.709e-03 ... -3.340e-07  2.860e-06]]
       r: [-5.360e+04 -8.331e+03 ...  7.282e-03  1.019e+00]
     qtf: [ 8.671e-02 -6.860e-03 ...  2.431e-08  4.619e-09]

In [ ]:
# in test.py:
# Debugging of hydraulic convergence
# Load class instance at crash point
net_id = 'Profile 3'
Kvleak_bool = True
dis_steps = 125
c = 50
N = 26811
pump_type = 'curve'

if N is not None:
    file_name = f"{net_id}_N={N}_Kvleak={Kvleak_bool}_hsteps={dis_steps}_pump={c}kPa_{pump_type}.pkl"
else: 
    file_name = f"{net_id}_Kvleak={Kvleak_bool}_hsteps={dis_steps}_pump={c}kPa_{pump_type}.pkl"

    
# base_dir = os.path.dirname(__file__)
base_dir = os.path.abspath('')

file = os.path.join(base_dir, 'debug', file_name)

with open(file, 'rb') as f:
    state = pickle.load(f)

net = Network.__new__(Network)  # create instance without calling __init__
net.__dict__.update(state)

theta = net.theta

print(file_name)

start = time.time()
print(f' start {datetime.now()}')
optimization_run(theta_1=theta[0], theta_2=theta[1], theta_3=theta[2], theta_4=theta[3], theta_5=theta[4], theta_6=theta[5], profile = 'Profile 3', opt = True)
end = time.time()
print(f"Execution time: {end - start} seconds")
